# LangChain 调用私有化 ChatGLM 模型

## LCEL 实现单轮对话

In [ ]:
from langchain_community.llms import ChatGLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# ChatGLM 私有化部署的 Endpoint URL
endpoint_url = "http://127.0.0.1:8001"

In [ ]:
# 实例化 ChatGLM 大模型
llm = ChatGLM(
    endpoint_url=endpoint_url,
    max_token=80000,
    history=[
        ["你是一个专业的销售顾问", "欢迎问我任何问题。"]
    ],
    top_p=0.9,
    model_kwargs={"sample_model_args": False},
)

In [ ]:
# 提示词模板
template = """{question}"""
prompt = PromptTemplate(template=template, input_variables=["question"])

In [ ]:
# 使用 LCEL（LangChain Expression Language）构建链
llm_chain = prompt | llm | StrOutputParser()

In [ ]:
# 使用 invoke 方法替代已废弃的 run 方法
llm_chain.invoke({"question": "你们衣服怎么卖？"})

## 带记忆功能的聊天对话（Conversation with Memory）

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [ ]:
# 会话历史存储
chat_history_store = {}

def get_session_history(session_id: str):
    """根据 session_id 获取或创建会话历史"""
    if session_id not in chat_history_store:
        chat_history_store[session_id] = ChatMessageHistory()
    return chat_history_store[session_id]

# 创建带有历史消息占位符的提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的销售顾问，可以回答关于衣服销售的问题。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 构建链
chain = prompt | llm | StrOutputParser()

# 使用 RunnableWithMessageHistory 包装链以支持会话历史
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [ ]:
# 使用 invoke 方法进行对话，需要传入 session_id 配置
conversation.invoke(
    {"input": "你们衣服怎么卖？"},
    config={"configurable": {"session_id": "session1"}}
)

In [ ]:
# 继续对话，使用相同的 session_id 以保持上下文
conversation.invoke(
    {"input": "有哪些款式？"},
    config={"configurable": {"session_id": "session1"}}
)

In [ ]:
# 继续对话，AI 会记住之前的内容
conversation.invoke(
    {"input": "休闲装男款都有啥？"},
    config={"configurable": {"session_id": "session1"}}
)